# **Phase 1 – Setup Environment**

In [ ]:
!pip install -U sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 49.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# **STEP 2 – Create Sample Client Document**

In [ ]:
sample_text = """
KOP Studios Contract:
Project Type: Portfolio Website
Budget: 120000 INR
Deadline: 45 days
Payment Terms: 50% advance, 50% after completion.

Aishwarya Design Agreement:
Project Type: Branding + Portfolio
Budget: 85000 INR
Deadline: 30 days
Payment Terms: Full payment after approval.
"""

with open("client_docs.txt", "w") as f:
    f.write(sample_text)

# **STEP 3 – Split Text into Chunks**

In [ ]:
def split_text(text, chunk_size=200):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

with open("client_docs.txt", "r") as f:
    text = f.read()

chunks = split_text(text)
chunks

['\nKOP Studios Contract:\nProject Type: Portfolio Website\nBudget: 120000 INR\nDeadline: 45 days\nPayment Terms: 50% advance, 50% after completion.\n\nAishwarya Design Agreement:\nProject Type: Branding + Port',
 'folio\nBudget: 85000 INR\nDeadline: 30 days\nPayment Terms: Full payment after approval.\n']

# **STEP 4 – Create Embeddings**

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

chunk_embeddings = embedding_model.encode(chunks)
chunk_embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[-4.77016754e-02,  5.17898798e-02,  7.00157182e-03,
        -3.57035659e-02,  1.44192940e-02,  3.94677669e-02,
        -1.65234413e-02, -4.35027818e-04,  4.08405215e-02,
         7.46849328e-02, -2.14283485e-02, -6.96223304e-02,
        -2.09130831e-02, -7.22010329e-04, -2.17782110e-02,
        -1.53471865e-02,  1.66039485e-02, -1.18724160e-01,
         5.16817905e-02, -8.51021428e-03, -4.18331586e-02,
        -3.81959789e-02,  2.09592897e-02, -4.62061912e-02,
         3.84052843e-02, -3.45087647e-02,  5.19096442e-02,
         1.02878518e-01,  4.12839092e-03, -1.28644649e-02,
        -3.07637043e-02,  9.66865718e-02,  7.77345523e-02,
        -5.69420420e-02,  5.55615239e-02,  8.75018910e-02,
        -3.87331508e-02, -5.57805449e-02, -3.68407443e-02,
         5.56393713e-03, -4.11384366e-02, -1.14965497e-03,
        -1.65556800e-02,  2.99981646e-02,  6.88920096e-02,
        -3.55599336e-02, -8.78455490e-02, -1.32471304e-02,
        -5.59504004e-03,  1.22122252e-02, -1.90793276e-0

# **STEP 5 – Store in FAISS**

In [ ]:
import faiss
import numpy as np

In [ ]:
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

In [ ]:
index.add(np.array(chunk_embeddings))

# **STEP 6 – Retrival Function**

In [ ]:
def retrieve(query, top_k=1):
  query_embedding = embedding_model.encode([query])
  distances, indices = index.search(np.array(query_embedding), top_k)
  return [chunks[i] for i in indices[0]]

In [ ]:
query = "What is the budget of KOP Studios?"
retrieve(query)

['\nKOP Studios Contract:\nProject Type: Portfolio Website\nBudget: 120000 INR\nDeadline: 45 days\nPayment Terms: 50% advance, 50% after completion.\n\nAishwarya Design Agreement:\nProject Type: Branding + Port']

# **STEP 7 – Add Local LLM for Answer Generation

In [ ]:
from transformers import pipeline

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
def generate_answer(query):
  context = retrieve(query)[0]
  prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
  response = generator(prompt, max_length=150, num_return_sequences=1)
  return response[0]["generated_text"]

In [ ]:
generate_answer("What is the budget for AishwaryaDesign?")

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Context: \nKOP Studios Contract:\nProject Type: Portfolio Website\nBudget: 120000 INR\nDeadline: 45 days\nPayment Terms: 50% advance, 50% after completion.\n\nAishwarya Design Agreement:\nProject Type: Branding + Port\n\nQuestion: What is the budget for AishwaryaDesign?\nAnswer: \xa0\xa0\n\xa0\nThe company is a large company in India and is working hard to support our growing talent in the field of retail and digitalisation. The company is based in India and is working hard to support our growing talent in the field of retail and digitalisation. The company is a large company in India and is working hard to support our growing talent in the field of retail and digitalisation. The company is a large company in India and is working hard to support our growing talent in the field of retail and digitalisation. The company is a large company in India and is working hard to support our growing talent in the field of retail and digitalisation. The company is a large company in India and is w